## 1. API Data Collection (Jamendo)

In [1]:
import requests
import pandas as pd

url = "https://api.jamendo.com/v3.0/tracks/"
params = {
    "client_id": "31ffae09",
    "format": "json",
    "limit": 200,
    "include": "musicinfo",
}

response = requests.get(url, params=params)
data = response.json()
print(data["headers"])

{'status': 'success', 'code': 0, 'error_message': '', 'warnings': '', 'results_count': 200, 'next': 'https://api.jamendo.com/v3.0/tracks?client_id=31ffae09&format=json&limit=200&include=musicinfo&offset=200'}


**Exploration of mood tags.** Only a small share of tracks have mood tags and "happy" is rare, so we collect tracks without a mood filter and take the mood from the vartags field.

In [2]:
all_tags = []

for track in data["results"]:
    vartags = track["musicinfo"]["tags"]["vartags"]
    all_tags.extend(vartags)

print("Total mood tags:", len(all_tags))
print(pd.Series(all_tags).value_counts().head(20))

Total mood tags: 59
neutral      17
peaceful     12
acappella     9
chorale       5
undefined     3
engage        3
spiritual     2
sad           2
action        1
energetic     1
emotional     1
happy         1
religious     1
enfants       1
Name: count, dtype: int64


In [3]:
import time

all_tracks = []

for offset in range(0, 1000, 200):
    params = {
        "client_id": "31ffae09",
        "format": "json",
        "limit": 200,
        "offset": offset,
        "include": "musicinfo",
        "order": "popularity_total",
        "datebetween": "2021-01-01_2025-12-31",
    }

    for attempt in range(3):
        response = requests.get(url, params=params)
        results = response.json()["results"]
        if len(results) > 0:
            break
        time.sleep(2)

    all_tracks.extend(results)
    print("offset", offset, "->", len(results), "tracks")
    time.sleep(1)

print("Total:", len(all_tracks))

offset 0 -> 200 tracks
offset 200 -> 200 tracks
offset 400 -> 200 tracks
offset 600 -> 200 tracks
offset 800 -> 200 tracks
Total: 1000


In [4]:
rows = []

for track in all_tracks:
    info = track["musicinfo"]
    tags = info["tags"]
    rows.append({
        "track_id": track["id"],
        "track_name": track["name"],
        "artist": track["artist_name"],
        "release_date": track["releasedate"],
        "duration_sec": track["duration"],
        "speed": info["speed"],
        "vocal_instrumental": info["vocalinstrumental"],
        "genres": ", ".join(tags["genres"]),
        "mood": tags["vartags"][0] if tags["vartags"] else None,
    })

jamendo_df = pd.DataFrame(rows)
jamendo_df.to_csv("jamendo_tracks.csv", index=False)
print(jamendo_df.shape)
jamendo_df.head()

(1000, 9)


,track_id,track_name,artist,release_date,duration_sec,speed,vocal_instrumental,genres,mood
0,1932670,RED LIGHT,Egor Budennyy,2022-03-17,190,medium,vocal,"pop, disco",groovy
1,2034080,Music Soothe - Meditation Music,StimiBeats,2023-06-01,216,low,instrumental,"newage, ambient",relaxing
2,1886257,Alone,Color Out,2021-09-20,245,medium,vocal,"emo, poprock",emotional
3,1876086,Love of My Life,Ben Lvcas,2021-08-04,161,high,vocal,"edm, pop, electronic",energetic
4,1826189,Not the One to Say (I Told You So) (24-bit Mas...,Cole Powell,2021-01-23,235,medium,vocal,"pop, jazz, singersongwriter",love


## 2. Web Scraping (Wikipedia Billboard Hot 100)

In [5]:
from bs4 import BeautifulSoup

headers = {"User-Agent": "KBTU student project (educational use)"}
billboard_rows = []

for year in range(2021, 2026):
    page_url = f"https://en.wikipedia.org/wiki/List_of_Billboard_Hot_100_number_ones_of_{year}"
    page = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(page.text, "html.parser")

    for table in soup.find_all("table", class_="wikitable"):
        header_text = table.find("tr").get_text()
        if "Song" in header_text and "Issue date" in header_text:
            break

    for row in table.find_all("tr")[1:]:
        cells = row.find_all(["th", "td"])
        if len(cells) >= 5:
            billboard_rows.append({
                "year": year,
                "issue_date": cells[1].get_text(strip=True),
                "song": cells[2].get_text(" ",strip=True).strip('"'),
                "artist": cells[3].get_text(" ",strip=True),
                "weeks_at_no1": int(cells[2].get("rowspan", 1)),
            })

    print(year, page.status_code)
    time.sleep(1)

billboard_df = pd.DataFrame(billboard_rows)
billboard_df.to_csv("billboard_hits.csv", index=False)
print(billboard_df.shape)
billboard_df.head(10)

2021 200
2022 200
2023 200
2024 200
2025 200
(109, 5)


,year,issue_date,song,artist,weeks_at_no1
0,2021,January 2,All I Want for Christmas Is You,Mariah Carey,1
1,2021,January 9,Mood,24kGoldn featuring Iann Dior,2
2,2021,January 23,Drivers License,Olivia Rodrigo,8
3,2021,March 20,What's Next,Drake,1
4,2021,March 27,Up,Cardi B,1
5,2021,April 3,Peaches,Justin Bieber featuring Daniel Caesar and Giveon,1
6,2021,April 10,Montero (Call Me by Your Name),Lil Nas X,1
7,2021,April 17,Leave the Door Open,Silk Sonic ( Bruno Mars and Anderson .Paak ) [...,1
8,2021,April 24,Rapstar,Polo G,2
9,2021,May 8,Save Your Tears,The Weeknd and Ariana Grande,2


## 3. Data Cleaning

In [6]:
jam = pd.read_csv("jamendo_tracks.csv")
bb = pd.read_csv("billboard_hits.csv")

print("Missing values in Jamendo:\n", jam.isna().sum())


jam = jam.drop_duplicates(subset="track_id")
jam = jam.dropna(subset=["mood"])
jam[["genres", "speed", "vocal_instrumental"]] = jam[["genres", "speed", "vocal_instrumental"]].fillna("unknown")
jam["release_date"] = pd.to_datetime(jam["release_date"])
jam["year"] = jam["release_date"].dt.year
jam["duration_sec"] = jam["duration_sec"].astype(int)
jam = jam.rename(columns={"track_name": "jamendo_track", "artist": "jamendo_artist"})


for col in ["song", "artist", "issue_date"]:
    bb[col] = bb[col].str.replace(r"\[.*?\]", "", regex=True).str.strip()
bb = bb.drop_duplicates()
bb["issue_date"] = pd.to_datetime(bb["issue_date"] + " " + bb["year"].astype(str),
                                  format="%B %d %Y", errors="coerce")
bb = bb.rename(columns={"song": "billboard_song", "artist": "billboard_artist"})

print("Jamendo after cleaning:", jam.shape)
print("Billboard after cleaning:", bb.shape)
print(jam.dtypes)
print("Missing values after cleaning:\n", jam.isna().sum())

Missing values in Jamendo:
 track_id               0
track_name             0
artist                 0
release_date           0
duration_sec           0
speed                 60
vocal_instrumental     4
genres                 2
mood                   2
dtype: int64
Jamendo after cleaning: (998, 10)
Billboard after cleaning: (109, 5)
track_id                       int64
jamendo_track                    str
jamendo_artist                   str
release_date          datetime64[us]
duration_sec                   int64
speed                            str
vocal_instrumental               str
genres                           str
mood                             str
year                           int32
dtype: object
Missing values after cleaning:
 track_id              0
jamendo_track         0
jamendo_artist        0
release_date          0
duration_sec          0
speed                 0
vocal_instrumental    0
genres                0
mood                  0
year                  0
dtype: in

## 4. Merging

In [7]:
bb_year = bb.groupby("year").agg(
    no1_songs=("billboard_song", "count"),
    avg_weeks_at_no1=("weeks_at_no1", "mean"),
    max_weeks_at_no1=("weeks_at_no1", "max"),
).reset_index()
print(bb_year)

merged = pd.merge(jam, bb_year, on="year", how="left")
merged.to_csv("merged_music.csv", index=False)
bb.to_csv("billboard_hits_clean.csv", index=False)

print(merged.shape)
merged.head()

   year  no1_songs  avg_weeks_at_no1  max_weeks_at_no1
0  2021         24          2.041667                 8
1  2022         20          2.650000                 6
2  2023         26          1.884615                10
3  2024         24          2.166667                14
4  2025         15          3.466667                13
(998, 13)


,track_id,jamendo_track,jamendo_artist,release_date,duration_sec,speed,vocal_instrumental,genres,mood,year,no1_songs,avg_weeks_at_no1,max_weeks_at_no1
0,1932670,RED LIGHT,Egor Budennyy,2022-03-17,190,medium,vocal,"pop, disco",groovy,2022,20,2.650000,6
1,2034080,Music Soothe - Meditation Music,StimiBeats,2023-06-01,216,low,instrumental,"newage, ambient",relaxing,2023,26,1.884615,10
2,1886257,Alone,Color Out,2021-09-20,245,medium,vocal,"emo, poprock",emotional,2021,24,2.041667,8
3,1876086,Love of My Life,Ben Lvcas,2021-08-04,161,high,vocal,"edm, pop, electronic",energetic,2021,24,2.041667,8
4,1826189,Not the One to Say (I Told You So) (24-bit Mas...,Cole Powell,2021-01-23,235,medium,vocal,"pop, jazz, singersongwriter",love,2021,24,2.041667,8
